In [2]:
### 資料清洗
"""
Data Cleaning Pipeline for NBA Salary Valuation System
專注於：名稱正規化、邊緣球員截斷、比率型特徵(%)補零、異常值處理
"""

import pandas as pd
import numpy as np
import re
import logging
import unicodedata

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 資料清洗函數
# 1. clean_player_names: 清洗球員名稱
# 2. clean_stats_data: 清洗 B-Ref 統計數據
# 3. clean_contract_data: 清洗 Spotrac 合約數據
# 4. convert_imperial_to_metric: 將體測指標從美制 (英吋/磅) 轉換為公制 (公分/公斤) (for physical attributes)
# 5. validate_data_quality: 資料品質檢驗器

def clean_player_names(df: pd.DataFrame, name_col: str = 'Player') -> pd.DataFrame:
    df = df.copy()
    def normalize_name(name):
        if pd.isna(name): return name
        name = str(name).lower()
        
        # ==========================================
        # 🚨 [亂碼修復] 攔截東歐姓氏常出現的亂碼特徵
        # ==========================================
        # 將錯亂的 viä (vić), kiä (kić), äiä (čić) 強制換回正常的 c
        name = name.replace('viä', 'vic')
        name = name.replace('kiä', 'kic')
        name = name.replace('äiä', 'cic')
        
        # 處理 Dario Šarić 常見的開頭亂碼 (å ariä -> saric)
        name = name.replace('å ', 's ')
        
        # 去除重音符號與亂碼
        name = ''.join(c for c in unicodedata.normalize('NFD', name) if unicodedata.category(c) != 'Mn')
        name = re.sub(r'[^\w\s]', '', name)
        name = re.sub(r' (jr|sr|ii|iii|iv)$', '', name)
        return name.strip()

    df[name_col] = df[name_col].apply(normalize_name)
    
    # 擴充特例字典
    name_mapping = {
        'nicolas claxton': 'nic claxton',
        'marcus morris sr': 'marcus morris',
        'kelly oubre': 'kelly oubre jr',
        'nene hilario': 'nene',
        'luc richard mbah a moute': 'luc mbah a moute',
        'marcus georgeshunt': 'marcus georges hunt',
        'dario aria': 'dario saric',
        'luka donaia': 'luka doncic',
        'ishmael smith': 'ish smith',
        'jose barea': 'jj barea',
        'louis williams': 'lou williams',
        'mohamed bamba': 'mo bamba',
        'ishmail wainright': 'ish wainright',
        'herb jones': 'herbert jones',
        'juancho hernangomez': 'juan hernangomez',
        'vincent poirier': 'vince poirier',
        'pj dozier': 'p j dozier',
    }
    df[name_col] = df[name_col].replace(name_mapping)
    return df

def resolve_traded_players(df: pd.DataFrame) -> pd.DataFrame:
    """
    處理季中交易球員 (2TM, 3TM, TOT)：
    1. 抓出該賽季的總計數據行 (2TM/TOT)
    2. 將 Team 替換成該季效力的最後一支球隊
    3. 刪除其餘的個別球隊拆分數據行
    """
    df = df.copy()
    
    # Basketball-Reference 常見的「多支球隊」標籤 (NBA 歷史單季最多換過 5 支球隊)
    multi_team_labels = ['TOT', '2TM', '3TM', '4TM', '5TM']
    
    # 建立一個 list 來儲存我們最終要保留的 index
    indices_to_keep = []
    
    # 透過 (Player, year) 進行分組，確保是在處理同一個人在同一個賽季的數據
    for (player, year), group in df.groupby(['Player', 'year']):
        # 找出是否有「總計行」 (例如 Team 是 2TM 或 TOT)
        tot_rows = group[group['Team'].isin(multi_team_labels)]
        
        if not tot_rows.empty:
            # 取得這筆總計行的 index
            tot_idx = tot_rows.index[0]
            
            # 找出該名球員在這個賽季效力的所有「真實球隊」紀錄 (排除 TOT / 2TM)
            team_rows = group[~group['Team'].isin(multi_team_labels)]
            
            if not team_rows.empty:
                # 因為 B-Ref 的預設排序是：2TM -> 第一支球隊 -> 第二支球隊 -> 最後一支球隊
                # 所以我們抓取 team_rows 的「最後一筆資料 (iloc[-1])」，這就是他的季末球隊！
                final_team = team_rows['Team'].iloc[-1]
                
                # 把總計行的 Team 替換成這支季末球隊
                df.loc[tot_idx, 'Team'] = final_team
            
            # 只保留這筆更新過球隊的「總計行」，其餘的拆分行不加入保留清單 (等於被刪除)
            indices_to_keep.append(tot_idx)
            
        else:
            # 如果沒有轉隊紀錄 (一般球員)，直接把所有 index 原封不動加進保留清單
            indices_to_keep.extend(group.index.tolist())
            
    # 根據保留下來的 index 重新篩選 DataFrame，並重置 index 讓表變乾淨
    df_cleaned = df.loc[indices_to_keep].reset_index(drop=True)
    
    return df_cleaned

def clean_stats_data(df: pd.DataFrame, min_games: int = 10) -> pd.DataFrame:
    """專屬 B-Ref 統計數據的清洗邏輯"""
    df = df.copy()
    
    # 1. 清洗球員名稱與格式
    df = clean_player_names(df, 'Player')
    if 'Player' in df.columns:
        df = df[df['Player'] != 'Player']

    # 確保 year 欄位絕對是整數
    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').fillna(0).astype(int)

    # 2. 在這裡執行季中交易球員的合併與清洗！
    df = resolve_traded_players(df)

    # 3. 數值轉型與後續處理
    id_cols = ['Player', 'Team', 'Pos', 'Awards', 'Season', 'Type', 'year']
    numeric_cols = [col for col in df.columns if col not in id_cols]
    
    for col in numeric_cols:
        if col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].astype(str).str.replace(r'[\*\,]', '', regex=True)
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 4. 邊緣球員截斷 (Thresholding)
    if 'G' in df.columns:
        initial_count = len(df)
        df = df[df['G'] >= min_games]
        dropped_count = initial_count - len(df)
        logger.info(f"移除了 {dropped_count} 筆出賽少於 {min_games} 場的雜訊數據。")

    # 5. 處理比率型數據的 0/0 問題 (例如 3P%, FT%)
    rate_cols = [col for col in df.columns if '%' in col]
    if rate_cols:
        df[rate_cols] = df[rate_cols].fillna(0)
        logger.info(f"已將比率型欄位 {rate_cols} 的 NaN 補為 0。")

    # 針對非比率型的其餘數值欄位（如上場時間太少導致的進階數據缺失），以中位數填補或維持 NaN 視後續合併而定
    # 這裡暫時保留其他特徵的 NaN，交由後續 Xgb 模型自動處理或在 Merge 階段插補。

    logger.info("統計數據 (Stats) 清洗完成。")
    return df


def clean_contract_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    專屬 Spotrac 合約數據的清洗邏輯
    確保目標變數 (Cap_Pct) 與年限 (YRS) 在合理範圍
    """
    df = df.copy()

    # 1. 名稱正規化
    df = clean_player_names(df, 'Player')

    # 2. 確保年份格式
    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    # 3. 目標變數清理 (Cap_Pct 應該介於 0 到 0.40 之間，因為頂薪最高為 35% 左右)
    if 'Cap_Pct' in df.columns:
        df['Cap_Pct'] = pd.to_numeric(df['Cap_Pct'], errors='coerce')
        # 若爬蟲抓到異常的百分比(如 500%)，將其限制在合理天花板
        df['Cap_Pct'] = df['Cap_Pct'].clip(lower=0, upper=0.40)
        # 移除沒有 Cap_Pct 的無效合約
        df = df.dropna(subset=['Cap_Pct'])

    # 4. 確保合約年限在 1-5 年 (CBA 規定極限)
    if 'YRS' in df.columns:
        df['YRS'] = pd.to_numeric(df['YRS'], errors='coerce').clip(lower=1, upper=5)

    # 5. 確保鳥權/續約旗標為 0 或 1
    if 'is_retained' in df.columns:
        df['is_retained'] = pd.to_numeric(df['is_retained'], errors='coerce').fillna(0).astype(int)
        df['is_retained'] = df['is_retained'].clip(lower=0, upper=1)

    logger.info("合約數據 (Contracts) 清洗完成。")
    return df


def validate_data_quality(df: pd.DataFrame, dataset_name: str = "Dataset") -> dict:
    """資料品質檢驗器"""
    validation_results = {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'missing_values': df.isnull().sum().to_dict(),
        'duplicate_rows': df.duplicated().sum()
    }

    missing_total = df.isnull().sum().sum()
    logger.info(f"[{dataset_name} 驗證] 總筆數: {len(df)} | 總欄位: {len(df.columns)} | 總缺失值: {missing_total}")

    return validation_results

### 1. Contract

In [53]:
contract_df = pd.read_csv('../../data/raw/Contract.csv')
contract_df = clean_player_names(contract_df)
contract_df = clean_contract_data(contract_df)
contract_df.to_csv('../../data/processed/contract_data.csv', index=False)
validate_data_quality(contract_df, 'Contract.csv')

2026-06-04 23:53:20,965 - INFO - 合約數據 (Contracts) 清洗完成。
2026-06-04 23:53:20,970 - INFO - [Contract.csv 驗證] 總筆數: 977 | 總欄位: 5 | 總缺失值: 0


{'total_rows': 977,
 'total_columns': 5,
 'missing_values': {'Player': 0,
  'YRS': 0,
  'is_retained': 0,
  'year': 0,
  'Cap_Pct': 0},
 'duplicate_rows': np.int64(0)}

### 2. Stats regular 常規賽

In [55]:
stats_reg_df = pd.read_csv('../../data/raw/Stats_reg_2014_2024.csv')
stats_reg_df = clean_player_names(stats_reg_df)
stats_reg_df = clean_stats_data(stats_reg_df)
stats_reg_df.to_csv('../../data/processed/Stats_reg_2014_2024_cleaned.csv', index=False)
validate_data_quality(stats_reg_df, "Reg Stats")

2026-06-04 23:53:38,535 - INFO - 移除了 667 筆出賽少於 10 場的雜訊數據。
2026-06-04 23:53:38,538 - INFO - 已將比率型欄位 ['FG%', '3P%', '2P%', 'eFG%', 'FT%', 'TS%', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%'] 的 NaN 補為 0。
2026-06-04 23:53:38,538 - INFO - 統計數據 (Stats) 清洗完成。
2026-06-04 23:53:38,633 - INFO - [Reg Stats 驗證] 總筆數: 5135 | 總欄位: 50 | 總缺失值: 0


{'total_rows': 5135,
 'total_columns': 50,
 'missing_values': {'Player': 0,
  'Age': 0,
  'Team': 0,
  'Pos': 0,
  'G': 0,
  'MP': 0,
  'FG': 0,
  'FGA': 0,
  'FG%': 0,
  '3P': 0,
  '3PA': 0,
  '3P%': 0,
  '2P': 0,
  '2PA': 0,
  '2P%': 0,
  'eFG%': 0,
  'FT': 0,
  'FTA': 0,
  'FT%': 0,
  'ORB': 0,
  'DRB': 0,
  'TRB': 0,
  'AST': 0,
  'STL': 0,
  'BLK': 0,
  'TOV': 0,
  'PF': 0,
  'PTS': 0,
  'year': 0,
  'Type': 0,
  'PER': 0,
  'TS%': 0,
  '3PAr': 0,
  'FTr': 0,
  'ORB%': 0,
  'DRB%': 0,
  'TRB%': 0,
  'AST%': 0,
  'STL%': 0,
  'BLK%': 0,
  'TOV%': 0,
  'USG%': 0,
  'OWS': 0,
  'DWS': 0,
  'WS': 0,
  'WS/48': 0,
  'OBPM': 0,
  'DBPM': 0,
  'BPM': 0,
  'VORP': 0},
 'duplicate_rows': np.int64(0)}

### 3. Stats Playoff 季後賽

In [56]:
stats_playoff_df = pd.read_csv('../../data/raw/Stats_playoffs_2014_2024.csv')
stats_playoff_df = clean_player_names(stats_playoff_df)
stats_playoff_df = clean_stats_data(stats_playoff_df, min_games=2)
stats_playoff_df.to_csv('../../data/processed/Stats_playoffs_2014_2024_cleaned.csv', index=False)
validate_data_quality(stats_playoff_df, "Playoffs Stats")

2026-06-04 23:53:41,348 - INFO - 移除了 128 筆出賽少於 2 場的雜訊數據。
2026-06-04 23:53:41,350 - INFO - 已將比率型欄位 ['FG%', '3P%', '2P%', 'eFG%', 'FT%', 'TS%', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%'] 的 NaN 補為 0。
2026-06-04 23:53:41,350 - INFO - 統計數據 (Stats) 清洗完成。
2026-06-04 23:53:41,400 - INFO - [Playoffs Stats 驗證] 總筆數: 2251 | 總欄位: 50 | 總缺失值: 60


{'total_rows': 2251,
 'total_columns': 50,
 'missing_values': {'Player': 0,
  'Age': 0,
  'Team': 0,
  'Pos': 0,
  'G': 0,
  'MP': 0,
  'FG': 0,
  'FGA': 0,
  'FG%': 0,
  '3P': 0,
  '3PA': 0,
  '3P%': 0,
  '2P': 0,
  '2PA': 0,
  '2P%': 0,
  'eFG%': 0,
  'FT': 0,
  'FTA': 0,
  'FT%': 0,
  'ORB': 0,
  'DRB': 0,
  'TRB': 0,
  'AST': 0,
  'STL': 0,
  'BLK': 0,
  'TOV': 0,
  'PF': 0,
  'PTS': 0,
  'year': 0,
  'Type': 0,
  'PER': 0,
  'TS%': 0,
  '3PAr': 30,
  'FTr': 30,
  'ORB%': 0,
  'DRB%': 0,
  'TRB%': 0,
  'AST%': 0,
  'STL%': 0,
  'BLK%': 0,
  'TOV%': 0,
  'USG%': 0,
  'OWS': 0,
  'DWS': 0,
  'WS': 0,
  'WS/48': 0,
  'OBPM': 0,
  'DBPM': 0,
  'BPM': 0,
  'VORP': 0},
 'duplicate_rows': np.int64(0)}

### 4. Physical Attributes 球員體測數據

In [17]:
physical_attributes = pd.read_csv('../../data/raw/kaggle_biometrics/all_seasons.csv')[['player_name', 'player_height', 'player_weight','draft_year']]
physical_attributes.rename(columns={'player_name':'Player','draft_year':'year'}, inplace=True)
physical_attributes = clean_player_names(physical_attributes)
physical_attributes.to_csv('../../data/processed/physical_attributes.csv', index=False)

2026-06-04 18:29:23,803 - INFO - 成功將 'Player' 欄位名稱正規化。


發現體測數據每個年份都是一樣的，但數據卻有所不同，故不採納。

### 5. Health 健康狀況

In [ ]:
health = pd.read_csv('../../data/raw/nba_attendance_injury_2010_2025.csv')
health['year'] = health['SEASON'].str[:4].astype(int) + 1
health['MIN_PER_GAME'] = health['MIN'] / health['GP']
health = health[['PLAYER_NAME', 'year', 'MIN_PER_GAME','ATTENDANCE_RATE','MAJOR_INJURY']]
health.rename(columns={'PLAYER_NAME':'Player'}, inplace=True)
health = clean_player_names(health)
health.to_csv('../../data/raw/nba_attendance_injury_2010_2025.csv', index=False)